In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Tue Mar 3

@source: https://gitlab.com/computational-neurologie/loss-of-plasticity 
@author: yaning
"""

import sys
import json
import torch
import pickle
import numpy as np
from tqdm import tqdm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import random_split

In [2]:
def calc_lyapunov(model, max_singularvalue):

	scale_factor = 1/estimate_dynamical_depth(model)

	return scale_factor * (max_singularvalue+1e-9).log()

def estimate_dynamical_depth(model):
	
	model.eval()
	
	# Initialize layer count
	layer_count = 0
	
	# Iterate through all modules in the model
	for layer in model.modules():
		# Check if the layer has learnable parameters
		if len(list(layer.parameters())) > 0:
			layer_count += 1

	return layer_count
    
def calc_singularvalues(model, inputs):

	# calc jacobian
	torch.cuda.empty_cache()
	jacobian_batch = torch.autograd.functional.jacobian(model, inputs, create_graph=True)

	# calc singular values
	flat_jacobian_batch = jacobian_batch.reshape(jacobian_batch.shape[0], jacobian_batch.shape[1], -1)
	svdvals = torch.linalg.svdvals(flat_jacobian_batch)

	# import pdb; pdb.set_trace()
	return svdvals



class CLossBackprop(object):
    def __init__(self, net, step_size=0.001, loss='mse', opt='sgd', beta_1=0.9, beta_2=0.999, weight_decay=0.0,
                 to_perturb=False, perturb_scale=0.1, device='cpu', momentum=0, alpha=0.1, lambda_goal=0, closs_every_N_steps=1):
        self.net = net
        self.to_perturb = to_perturb
        self.perturb_scale = perturb_scale
        self.device = device
        self.alpha = alpha
        self.lambda_goal = lambda_goal
        self.closs_every_N_steps = closs_every_N_steps

        print(f"This run uses alpha={self.alpha} and lambda_goal={self.lambda_goal} in CLoss. CLoss is applied every {closs_every_N_steps} steps.")

        # define the optimizer
        if opt == 'sgd':
            self.opt = optim.SGD(self.net.parameters(), lr=step_size, weight_decay=weight_decay, momentum=momentum)
        elif opt == 'adam':
            self.opt = optim.Adam(self.net.parameters(), lr=step_size, betas=(beta_1, beta_2),
                                  weight_decay=weight_decay)
        elif opt == 'adamW':
            self.opt = optim.AdamW(self.net.parameters(), lr=step_size, betas=(beta_1, beta_2),
                                   weight_decay=weight_decay)

        # define the loss function
        self.loss = loss
        self.loss_func = {'nll': F.cross_entropy, 'mse': F.mse_loss}[self.loss]

        # Placeholder
        self.previous_features = None
        self.singularvalues = None
        self.loss1 = None
        self.loss2 = None
        self.global_step = 0

    def learn(self, x, target):
        """
        Learn using one step of gradient-descent
        :param x: input
        :param target: desired output
        :return: loss
        """
        self.opt.zero_grad()
        output, features = self.net.predict(x=x)

        crossentropy_loss = self.loss_func(output, target)
        if self.global_step%self.closs_every_N_steps==0:
            try:
                singularvalues = calc_singularvalues(self.net, x[0,...]) # only calc for first sample
            except torch._C._LinAlgError:
                # import pdb; pdb.set_trace()
                singularvalues = torch.full(output[0,:][None,:].shape, torch.nan)
            lyapunov = calc_lyapunov(self.net, singularvalues.max())
            manual_scale = self.alpha
            lyapunov_loss = (self.lambda_goal-lyapunov)**2
            lyapunov_loss_scaled = lyapunov_loss * manual_scale
            # if crossentropy_loss>1e5: import pdb; pdb.set_trace()
            loss = crossentropy_loss + lyapunov_loss_scaled
            self.loss2 = lyapunov_loss_scaled.detach()
        else:
            loss = crossentropy_loss
            self.loss2 = torch.tensor(torch.nan)
        # loss = lyapunov_loss_scaled
        
        self.previous_features = features
        # self.singularvalues = singularvalues.detach()
        self.loss1 = crossentropy_loss.detach()

        loss.backward()

        # clip gradients
        torch.nn.utils.clip_grad_norm_(self.net.parameters(), max_norm=5.0)

        self.opt.step()
        self.global_step += 1
        if self.to_perturb:
            self.perturb()
        if self.loss == 'nll':
            return loss.detach(), output.detach()
        return loss.detach()

    def perturb(self):
        with torch.no_grad():
            for i in range(int(len(self.net.layers)/2)+1):
                self.net.layers[i * 2].bias +=\
                    torch.empty(self.net.layers[i * 2].bias.shape, device=self.device).normal_(mean=0, std=self.perturb_scale)
                self.net.layers[i * 2].weight +=\
                    torch.empty(self.net.layers[i * 2].weight.shape, device=self.device).normal_(mean=0, std=self.perturb_scale)


In [3]:
class ConvNet(nn.Module):
    def __init__(self, num_classes=2):
        """
        Convolutional Neural Network with 3 convolutional layers followed by 3 fully connected layers
        """
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 5)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.conv3 = nn.Conv2d(64, 128, 3)

        self.last_filter_output = 6 * 6
        self.num_conv_outputs = 128 * self.last_filter_output

        self.fc1 = nn.Linear(self.num_conv_outputs, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.pool = nn.MaxPool2d(2, 2)

        # architecture
        self.layers = nn.ModuleList()
        self.layers.append(self.conv1)
        self.layers.append(nn.ReLU())
        self.layers.append(self.conv2)
        self.layers.append(nn.ReLU())
        self.layers.append(self.conv3)
        self.layers.append(nn.ReLU())
        self.layers.append(self.fc1)
        self.layers.append(nn.ReLU())
        self.layers.append(self.fc2)
        self.layers.append(nn.ReLU())
        self.layers.append(self.fc3)

        self.act_type = 'relu'

    def predict(self, x):
        x1 = self.pool(self.layers[1](self.layers[0](x)))
        x2 = self.pool(self.layers[3](self.layers[2](x1)))
        x3 = self.pool(self.layers[5](self.layers[4](x2)))
        x3 = x3.view(-1, self.num_conv_outputs)
        x4 = self.layers[7](self.layers[6](x3))
        x5 = self.layers[9](self.layers[8](x4))
        x6 = self.layers[10](x5)
        return x6, [x1, x2, x3, x4, x5]
    
    # added this for SV calc
    def forward(self, x):
        x1 = self.pool(self.layers[1](self.layers[0](x)))
        x2 = self.pool(self.layers[3](self.layers[2](x1)))
        x3 = self.pool(self.layers[5](self.layers[4](x2)))
        x3 = x3.view(-1, self.num_conv_outputs)
        x4 = self.layers[7](self.layers[6](x3))
        x5 = self.layers[9](self.layers[8](x4))
        x6 = self.layers[10](x5)
        return x6

In [4]:
# load dataset
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),       
])

train_dataset = datasets.ImageFolder(
    root="/Users/compneuro1/Downloads/tiny-imagenet-200/train",
    transform=transform
)

In [6]:
net = ConvNet(num_classes=200)

learner = CLossBackprop(
			net=net,
			step_size=1e-3,
			opt="sgd",
			loss='nll',
			weight_decay=0,
			to_perturb = False,
			# to_perturb=(perturb_scale != 0),
			# perturb_scale=0.01,
            # step_size=[0.01,0.001],
			device="cpu",
			momentum=0.9,
			alpha=0.1,
			lambda_goal=0.0,
			closs_every_N_steps=20
		)

This run uses alpha=0.1 and lambda_goal=0.0 in CLoss. CLoss is applied every 20 steps.


In [10]:
learner.net.load_state_dict(torch.load("model_200c_60e_param1.pth", map_location="cpu"))

<All keys matched successfully>